In [ ]:
import datasets,huggingface_hub,os

In [ ]:
huggingface_hub.login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from datasets import load_dataset,concatenate_datasets
import numpy as np

In [ ]:
from datasets.inspect import get_dataset_config_info
dataset = load_dataset("Yelp/yelp_review_full")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})


In [ ]:
get_dataset_config_info("Yelp/yelp_review_full")

DatasetInfo(description='', citation='', homepage='', license='', features={'label': ClassLabel(names=['1 star', '2 star', '3 stars', '4 stars', '5 stars']), 'text': Value('string')}, post_processed=None, supervised_keys=None, builder_name='parquet', dataset_name='yelp_review_full', config_name='yelp_review_full', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=483892804, num_examples=650000, shard_lengths=None, dataset_name='yelp_review_full'), 'test': SplitInfo(name='test', num_bytes=37277438, num_examples=50000, shard_lengths=None, dataset_name='yelp_review_full')}, download_checksums={'hf://datasets/Yelp/yelp_review_full@c1f9ee939b7d05667af864ee1cb066393154bf85/yelp_review_full/train-00000-of-00001.parquet': {'num_bytes': 299436850, 'checksum': None}, 'hf://datasets/Yelp/yelp_review_full@c1f9ee939b7d05667af864ee1cb066393154bf85/yelp_review_full/test-00000-of-00001.parquet': {'num_bytes': 23515519, 'checksum': None}}, download_size=322952369, post_processing_size=N

In [ ]:
dataset = concatenate_datasets([dataset["train"], dataset["test"]])

In [ ]:
def tokenize(example):
    tokens = example["text"].split()
    return {
        "tokens": tokens,
        "length": len(tokens)
    }
dataset = dataset.map(tokenize)

Map:   0%|          | 0/700000 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['label', 'text', 'tokens', 'length'],
    num_rows: 700000
})

In [ ]:
lengths = np.array(dataset["length"])
print(lengths)
total_words_length = lengths.sum()
print(total_words_length)

[ 93 115  97 ... 105 138 156]
93878307


In [ ]:
single_word_count = np.sum(lengths == 1)
print(single_word_count)

355


In [ ]:
print(lengths.mean())


134.11186714285714


In [ ]:
from scipy.stats import skew
print(skew(lengths))

2.2343957557732184


In [ ]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
print(bert_tok)

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [ ]:
long_text = "hello " * 600  # 600 words

encoded = bert_tok(long_text)
print("Length without truncation:", len(encoded["input_ids"]))

encoded_truncated = bert_tok(long_text, truncation=True)
print("Length with truncation:", len(encoded_truncated["input_ids"]))

Token indices sequence length is longer than the specified maximum sequence length for this model (602 > 512). Running this sequence through the model will result in indexing errors


Length without truncation: 602
Length with truncation: 512


In [ ]:
sample = "This is a test sentence."
encoded = bert_tok(sample)

tokens = bert_tok.convert_ids_to_tokens(encoded["input_ids"])
print(tokens)

['[CLS]', 'this', 'is', 'a', 'test', 'sentence', '.', '[SEP]']


In [ ]:
from transformers import BertConfig, BertForMaskedLM

config = BertConfig()

model = BertForMaskedLM(config)

print("Config:",config)
print("Model:",model)

Config: BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

Model: BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encod

In [ ]:
word_embeddings = model.bert.embeddings.word_embeddings

print("Word embedding shape:", word_embeddings.weight.shape)

token_embedding_params = word_embeddings.weight.numel()

print("Token embedding parameters:", token_embedding_params)

Word embedding shape: torch.Size([30522, 768])
Token embedding parameters: 23440896


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)

Total parameters: 109514298


In [ ]:
config_long = BertConfig(max_position_embeddings=1024)
model_long = BertForMaskedLM(config_long)
params_long = sum(p.numel() for p in model_long.parameters())
print("Long context params:", params_long)

Long context params: 109907514


In [ ]:
print("Difference:", params_long - total_params)

Difference: 393216


In [ ]:
def chunk_examples(examples):
    # 1. Tokenize batch (1000 samples)
    tokenized = bert_tok(
        examples["text"],
        return_attention_mask=True,
        add_special_tokens=True,
        truncation=False
    )

    # 2. Concatenate all input_ids
    concatenated_input_ids = []
    concatenated_attention_mask = []

    for ids in tokenized["input_ids"]:
        concatenated_input_ids.extend(ids)

    for mask in tokenized["attention_mask"]:
        concatenated_attention_mask.extend(mask)

    # 3. Compute total length
    total_length = len(concatenated_input_ids)

    # 4. Drop remainder
    total_length = (total_length // 512) * 512

    # 5. Chunk into 512
    input_ids_chunks = [
        concatenated_input_ids[i : i + 512]
        for i in range(0, total_length, 512)
    ]

    attention_mask_chunks = [
        concatenated_attention_mask[i : i + 512]
        for i in range(0, total_length, 512)
    ]

    return {
        "input_ids": input_ids_chunks,
        "attention_mask": attention_mask_chunks,
    }

# Apply mapping with required batch size = 1000
ds_chunked = dataset.map(
    chunk_examples,
    batched=True,
    batch_size=1000,
    remove_columns=dataset.column_names,
)

print("Total samples in ds_chunked:", len(ds_chunked))

Map:   0%|          | 0/700000 [00:00<?, ? examples/s]

Total samples in ds_chunked: 246695


In [ ]:
ds_chunked

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 246695
})

In [ ]:
from datasets import DatasetDict

ds_split = ds_chunked.train_test_split(
    test_size=0.05,
    seed=42
)

train_ds = ds_split["train"]
test_ds = ds_split["test"]

In [ ]:
from transformers import DataCollatorForLanguageModeling, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.2
)

In [ ]:
from torch.utils.data import DataLoader

train_ds.set_format(type="torch")

dataloader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator
)

In [ ]:
batch = next(iter(dataloader))

input_ids = batch["input_ids"]
labels = batch["labels"]

In [ ]:
import torch

# Find first unmasked token
mask_positions = (labels == -100)

# Get first unmasked token id
unmasked_token_id = input_ids[mask_positions][0].item()

print("Unmasked token ID:", unmasked_token_id)

Unmasked token ID: 1010


In [ ]:
from transformers import BertConfig, BertForMaskedLM

config_small = BertConfig(
    num_hidden_layers=6,
    hidden_size=384,
    intermediate_size=1536
)

model_small = BertForMaskedLM(config_small)

In [ ]:
train_ds

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 234360
})

In [ ]:
train_ds.set_format(type="torch")

In [ ]:
from transformers import DataCollatorForLanguageModeling, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.2
)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

In [ ]:
import torch
from torch.optim import AdamW
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_small.to(device)

optimizer = AdamW(model_small.parameters(), lr=5e-5)

model_small.train()

for batch in tqdm(train_loader):
    batch = {k: v.to(device) for k, v in batch.items()}

    outputs = model_small(**batch)
    loss = outputs.loss

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

print("Final loss:", loss.item())

100%|██████████| 29295/29295 [1:59:17<00:00,  4.09it/s]


Final loss: 5.800103664398193
